# Nano-U Interactive Pipeline Decomposition 🔬🌱

Welcome to the **Nano-U Pipeline Breakdown**. This notebook is designed to let you run the Nano-U logic step-by-step, removing the abstraction of the CLI scripts (`train_standard.py`, `nas_search.py`) so you can strictly see what is going on under the hood.

We will interactively walk through:
1. **Loading Configuration and Data**
2. **Building the Neural Architecture Search (NAS) Space**
3. **Proxy Training and SVD Redundancy Computation**
4. **Distilling the Teacher into the Student**

--- 
## 1. Setup & Installation

First, we clone the repository, install the environment, and import the core tools.

In [ ]:
# Clone the repository (if running on Colab)
# !git clone https://github.com/federico-pizz/Nano-U.git
# %cd Nano-U
# !pip install -r requirements.txt

import os
from pathlib import Path
import tensorflow as tf
import numpy as np

# Suppress verbose TF warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print(f"TensorFlow Version: {tf.__version__}")

--- 
## 2. Loading Configuration & Data Pipeline

The core of the data loading happens in `src/data.py::make_dataset`. It uses TensorFlow's `tf.data.Dataset` mapping features to efficiently read, decode, and augment imagery on the CPU while the GPU computes the forward pass. We pull our normalization constants from `config/config.yaml`.

In [ ]:
from src.utils.config import load_config
from src.data import make_dataset

# 1. Load the global configuration
config = load_config('config/config.yaml')
data_cfg = config['data']

print("Global Normalize Mean:", data_cfg['normalization']['mean'])
print("Global Normalize Std:", data_cfg['normalization']['std'])

# 2. Find validation imagery in the processed_data folder
val_dir = Path(data_cfg['paths']['processed']['val']['img'])
val_mask_dir = Path(data_cfg['paths']['processed']['val']['mask'])

# Example of initializing the dataset if files were present:
if val_dir.exists():
    img_files = sorted(val_dir.glob('*.png'))
    mask_files = sorted(val_mask_dir.glob('*.png'))
    
    # make_dataset handles reading cv2 images and converting to tf.Tensor
    sample_ds = make_dataset(
        [str(f) for f in img_files[:10]], 
        [str(f) for f in mask_files[:10]], 
        batch_size=2, 
        shuffle=False
    )
    # Peek at the tensor shape (Batch, Height, Width, Channels)
    for x, y in sample_ds.take(1):
        print("\nDataset X shape:", x.shape)
        print("Dataset Y (mask) shape:", y.shape)

--- 
## 3. Dissecting the NAS Search Space

Before running a search, let's manually instantiate a **Nano-U** model utilizing the four distinct Block Primitives. 

The model architecture is defined by an array of integers `arch_seq`, where each integer designates a block type loaded from `get_block_map()`. Nano-U expects a 7-element array representing (3 Encoders, 1 Bottleneck, 3 Decoders).

In [ ]:
from src.models.builders import create_searchable_nano_u, get_block_map

# Let's print out the Mapping logic so we understand what integers mean what blocks:
print("---- BLOCK PRIMITIVES ----")
for idx, func in get_block_map().items():
    print(f"{idx} -> {func.__name__}")

# Manually define a sequence
# Example: All standard depthwise_sep_convs except for a triple_conv at the bottleneck
man_seq = [0, 0, 0, 1, 0, 0, 0]

manual_model = create_searchable_nano_u(
    input_shape=(48, 64, 3),
    filters=[16, 32, 64],    # Expanding channels in encoder
    bottleneck_filters=64,   # Channel count in the middle
    arch_seq=man_seq,
    name="nano_u_manual"
)

print(f"\nParam count of architecture {man_seq}: {manual_model.count_params():,}")

--- 
## 4. Understanding SVD Redundancy (`src/nas.py`)

During proxy training in the Evolutionary search, Nano-U penalizes models that have "high Redundancy." Redundant Convolution models learn the same features repeatedly (low rank representation).

We calculate this by taking the raw output (Activations) from the Convolutional layers, flattening them, executing a **Singular Value Decomposition (SVD)** on the covariance matrix, and checking the Condition Number / Rank. Here is the *under-the-hood* math:

In [ ]:
from src.nas import compute_layer_redundancy

# Let's mock a Batch of Feature Maps coming out of a convolution layer.
# Shape: (Batch=4, Height=24, Width=32, Channels=16)
mock_activations = tf.random.normal((4, 24, 32, 16))

# Send it to the SVD math function used in the NAS engine:
redundancy_info = compute_layer_redundancy(mock_activations)

print("SVD Redundancy Metrics on Mock Data:")
for key, val in redundancy_info.items():
    if isinstance(val, float):
        print(f" - {key}: {val:.4f}")
    else:
        print(f" - {key}: {val}")

print("\n-> If Random Data creates a Full Rank, then highly correlated real data will have a lower rank, indicating redundancy.")

--- 
## 5. Knowledge Distillation Process (`src/train.py`)

If normal `model.fit()` runs Standard Training, how does the Distillation work?

The code in `train_with_distillation` uses a custom `tf.GradientTape()` loop. It takes the **Teacher's soft probability outputs**, scales them by a `temperature`, and forces the student to match them using Mean Squared Error (MSE), while simultaneously mapping to the true Mask using Binary Crossentropy.

Let's simulate exactly what happens in ONE step of that loop:

In [ ]:
from src.models.builders import create_bu_net

# 1. Initialize the massive Teacher (BU-Net)
teacher = create_bu_net((48, 64, 3), filters=[64, 128, 256, 512, 1024])
teacher.trainable = False  # Freeze teacher
print(f"Teacher Params: {teacher.count_params():,}")

# 2. Initialize the tiny Student (Nano-U)
student = create_searchable_nano_u((48, 64, 3), filters=[16, 32, 64], bottleneck_filters=64, arch_seq=[0]*7)
print(f"Student Params: {student.count_params():,}")

# 3. Mock a single image and mask
x_batch = tf.random.normal((2, 48, 64, 3)) # Batch of 2 fake images
y_true = tf.random.uniform((2, 48, 64, 1), minval=0, maxval=2, dtype=tf.int32) # Fake Masks (0 or 1)
y_true = tf.cast(y_true, tf.float32)

# Set hyperparemters (From train_with_distillation)
alpha = 0.5         # 50% CE Loss, 50% Teacher MSE Loss
temperature = 4.0   # Soften the teacher logits heavily

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
bce_fn = tf.keras.losses.BinaryCrossentropy()
mse_fn = tf.keras.losses.MeanSquaredError()

# ---- THE CUSTOM TRAINING STEP ----
with tf.GradientTape() as tape:
    # Forward pass of Student
    student_logits = student(x_batch, training=True)
    
    # Forward pass of Teacher (No gradients)
    teacher_logits = teacher(x_batch, training=False)
    
    # Loss 1: True Labels (Hard Loss)
    loss_hard = bce_fn(y_true, student_logits)
    
    # Loss 2: Distillation (Soft Loss)
    # Note: Because the model outputs probabilities (sigmoid), taking log creates logits.
    # Then we apply Temperature to soften those probability distributions.
    student_soft = tf.math.log(student_logits + 1e-7) / temperature
    teacher_soft = tf.math.log(teacher_logits + 1e-7) / temperature
    loss_soft = mse_fn(teacher_soft, student_soft)
    
    # Final Combined Loss
    total_loss = (1 - alpha) * loss_hard + (alpha * loss_soft * (temperature ** 2))

# Apply Backpropagation to Student ONLY
gradients = tape.gradient(total_loss, student.trainable_variables)
optimizer.apply_gradients(zip(gradients, student.trainable_variables))

print(f"\nStep completed successfully.")
print(f"BCE Loss: {loss_hard:.4f}")
print(f"Distillation MSE Loss: {loss_soft:.4f}")
print(f"Total Weighted Loss: {total_loss:.4f}")